<a href="https://colab.research.google.com/github/oliver-morris/Dictionary-Language-Model/blob/main/NeuralNetZero.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Dictionary-Based LLM

Attempt at making a more computationally efficient LLM by leveraging dictionary-basedf learning.

# Imports

In [ ]:
!pip install -q transformers datasets sentencepiece nltk

In [ ]:
import random
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import nltk
from nltk.corpus import wordnet as wn
from transformers import AutoTokenizer

nltk.download("wordnet")
nltk.download("omw-1.4")

device = "cuda" if torch.cuda.is_available() else "cpu"

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


In [ ]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
MAX_LEN = 64

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

# Dataset

In [ ]:
samples = []

for syn in wn.all_synsets():
    word = syn.lemmas()[0].name().replace("_", " ")
    definition = syn.definition()
    examples = syn.examples()
    # Task 1
    samples.append(
        (
            f"define: {word}",
            definition
        )
    )

    # Task 2
    samples.append(
        (
            f"word: {definition}",
            word
        )
    )

    # Task 3
    if len(examples) > 0:
        samples.append(
            (
                f"example: {word}",
                examples[0]
            )
        )

random.shuffle(samples)
print("Training samples:", len(samples))

Training samples: 268241


In [ ]:
class DictDataset(Dataset):

    def __init__(self, pairs):

        self.data = pairs

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):

        inp, out = self.data[idx]

        enc = tokenizer(
            inp,
            padding="max_length",
            truncation=True,
            max_length=MAX_LEN,
            return_tensors="pt"
        )

        dec = tokenizer(
            out,
            padding="max_length",
            truncation=True,
            max_length=MAX_LEN,
            return_tensors="pt"
        )

        return (
            enc.input_ids.squeeze(),
            dec.input_ids.squeeze()
        )

dataset = DictDataset(samples)

loader = DataLoader(
    dataset,
    batch_size=32,
    shuffle=True
)

# Transformer

In [ ]:
VOCAB = tokenizer.vocab_size

class TinyDictionaryTransformer(nn.Module):

    def __init__(self):

        super().__init__()

        self.embedding = nn.Embedding(VOCAB,256)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=256,
            nhead=4,
            dim_feedforward=512,
            dropout=0.1,
            batch_first=True
        )

        self.encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=6
        )

        self.fc = nn.Linear(256,VOCAB)

    def forward(self,x):

        x = self.embedding(x)

        x = self.encoder(x)

        return self.fc(x)

model = TinyDictionaryTransformer().to(device)

# Training

In [ ]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=3e-4
)

criterion = nn.CrossEntropyLoss(ignore_index=0)

In [ ]:
EPOCHS = 5

for epoch in range(EPOCHS):

    model.train()

    total_loss = 0

    for x,y in loader:

        x = x.to(device)
        y = y.to(device)

        optimizer.zero_grad()

        pred = model(x)

        loss = criterion(
            pred.view(-1,VOCAB),
            y.view(-1)
        )

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    print(
        f"Epoch {epoch+1}",
        total_loss/len(loader)
    )

Epoch 1 5.982252210958126
Epoch 2 5.7686649547419
Epoch 3 5.638651579188288
Epoch 4 5.523109913554892
Epoch 5 5.4119693545528404


# Evaluating

In [ ]:
model.eval()

while True:

    text = input("\nEnter prompt: ")

    if text=="quit":
        break

    tokens = tokenizer(
        text,
        return_tensors="pt",
        padding="max_length",
        truncation=True,
        max_length=MAX_LEN
    ).input_ids.to(device)

    with torch.no_grad():

        out = model(tokens)

        ids = out.argmax(-1)

    print(
        tokenizer.decode(
            ids[0],
            skip_special_tokens=True
        )
    )